# Inference Time Study

In [ ]:
################################################## Initialize ##################################################

# Add the new path
import sys
new_path = "/home/michele/code/michele_mmdet3d/"
if not new_path in sys.path:
    sys.path.insert(1, new_path)

# Main variables
    # Boolean to print inputs/outputs of each stage
wanna_print_in_out = False
    # Home directory within ADE
home_dir = '/home/michele/code/'
    # Relative path from "home_dir" to the pointcloud and image files
path_from_home_to_pointcloud_files = "michele_mmdet3d/data/minerva_polimove/training/velodyne_reduced"
path_from_home_to_image_files = "michele_mmdet3d/data/minerva_polimove/training/image_2"
    # Path to the ".txt" file in "ImageSets", containing the list of validation files
val_list_txt_file = "/home/michele/code/michele_mmdet3d/data/minerva_polimove/ImageSets/val.txt"
    # Built-in inferencer boolean, to select the type of built model
boolean_built_in_inferencer = False



############################################# Time-related variables #############################################
import time
import torch
if not torch.cuda.is_available():
    raise MemoryError("\t\tCUDA not available: exiting...\n\n\n")

delta_preprocessing = []
delta_image_features = []
delta_point_fusion = []
delta_point_encoder = []
delta_postprocessing = []



################################################ Create the model ################################################
from mmdet3d.apis.inferencers import MultiModalityDet3DInferencer

# Build the model with the inferencer
if not boolean_built_in_inferencer:
    inferencer = MultiModalityDet3DInferencer(model="/home/michele/code/michele_mmdet3d/configs/minerva/MINERVA_mvxnet.py",
                                            weights="/home/michele/code/michele_mmdet3d/work_dirs/MINERVA_mvxnet/epoch_50.pth")
    model_cfg = inferencer.cfg
    model = inferencer.model



############################################### Automatic configs ###############################################
config_dictionary = {}

# For the loader LoadPointsFromFile
config_dictionary['LoadPointsFromFile'] = {
    'coord_type': model_cfg.val_dataloader.dataset.pipeline[0].coord_type,
    'load_dim': model_cfg.val_dataloader.dataset.pipeline[0].load_dim,
    'use_dim': model_cfg.val_dataloader.dataset.pipeline[0].use_dim
}

config_dictionary['Det3DDataPreprocessor'] = {
    'voxel': model_cfg.model.data_preprocessor.voxel,
    'voxel_type': model_cfg.model.data_preprocessor.voxel_type,
    'voxel_layer': model_cfg.model.data_preprocessor.voxel_layer,
    'mean': model_cfg.model.data_preprocessor.mean,
    'std': model_cfg.model.data_preprocessor.std,
    'bgr_to_rgb': model_cfg.model.data_preprocessor.bgr_to_rgb,
    'pad_size_divisor': model_cfg.model.data_preprocessor.pad_size_divisor
}

In [16]:
############################################### Very STARTING input ###############################################

# Read the names of the validation files from the ".txt" file in "ImageSets"
with open(val_list_txt_file, 'r') as file:
    val_file_names = sorted([line.strip() for line in file])



# Create the inputs
#   - See the details in "inference_and_model_FUSION.ipynb"
#   - For each input path to Pointcloud-Image-InfoFile
import os
inputs = []
for file_name in val_file_names:
    inputs.append([
        os.path.join(home_dir, path_from_home_to_pointcloud_files, file_name+".bin"),
        os.path.join(home_dir, path_from_home_to_image_files, file_name+".png")
    ])



# Print the output of this stage
if wanna_print_in_out:
    print("\nStarting values:")
    for element in inputs:
        print(element)

In [17]:
###################################################################################################################
###############################################  LoadPointsFromFile ###############################################
###############################################                     ###############################################
###############################################      ...and...      ###############################################
###############################################                     ###############################################
###############################################  LoadImageFromFile  ###############################################
###################################################################################################################

from mmdet3d.datasets.transforms.loading import LoadPointsFromFile
from mmcv.transforms.loading import LoadImageFromFile

# Initialize the loaders
loader_pointcloud = LoadPointsFromFile(
    coord_type=config_dictionary['LoadPointsFromFile']['coord_type'],
    load_dim=config_dictionary['LoadPointsFromFile']['load_dim'],
    use_dim=config_dictionary['LoadPointsFromFile']['use_dim']
)
loader_image = LoadImageFromFile()

# Print the input to this stage
if wanna_print_in_out:
    print("\nLoadPointsFromFile and LoadImageFromFile input:")
    for element in inputs:
        print(element)



# For cycle to also handle lists of inputs
for i in range(len(inputs)):
    
    # Prepare the string input for the loaders
    #   - The Pointcloud Loader needs two dictionaries, one nested into the other
    #   - The Image Loader needs just one dictionary
    inputs[i] = dict(
        lidar_points=dict(
            lidar_path=inputs[i][0]
        ),
        img_path=inputs[i][1]
    )

    # Actual modification of the dictionary
    loader_pointcloud(inputs[i])
    loader_image(inputs[i])



# Print the output of this stage
if wanna_print_in_out: 
    print("\nLoadPointsFromFile and LoadImageFromFile output:")
    for element in inputs:
        print(element)

In [18]:
############################################### Add METAINFO ###############################################

import mmengine
import numpy as np
import os.path as osp
from mmdet3d.structures.bbox_3d import Box3DMode, LiDARInstance3DBoxes

# Print the input of this stage
if wanna_print_in_out: 
    print("\nBefore adding metainfos:")
    for element in inputs:
        print(element)

# Get the METAINFO from the .pkl file (saved during create_data.py)
infos_path = "/home/michele/code/michele_mmdet3d/data/minerva_polimove/minerva_polimove_infos_val.pkl"
info_list = mmengine.load(infos_path)['data_list']



# Check the format of info_list, and make it match inputs
new_info_list = []
if not len(info_list) == len(inputs):
    # Copied from MultiModalityDet3DInferencer._inputs_to_list()
    for element in inputs:
        timestamp = element['img_path'].split('/')[-1].split('.')[0]
        for element in info_list:
            if str(element['sample_idx']) == str(timestamp):
                new_info_list.append(element)
    # Update the field "info_list"    
    info_list = new_info_list



# Get the right fields
cam_type = "CAM2"
for index, input in enumerate(inputs):
    data_info = info_list[index]
    img_path = data_info['images'][cam_type]['img_path']
    if isinstance(input['img'], str) and \
            osp.basename(img_path) != osp.basename(input['img']):
        raise ValueError(
            f'the info file of {img_path} is not provided.')
    cam2img = np.asarray(
        data_info['images'][cam_type]['cam2img'], dtype=np.float32)
    lidar2cam = np.asarray(
        data_info['images'][cam_type]['lidar2cam'],
        dtype=np.float32)
    if 'lidar2img' in data_info['images'][cam_type]:
        lidar2img = np.asarray(
            data_info['images'][cam_type]['lidar2img'],
            dtype=np.float32)
    else:
        lidar2img = cam2img @ lidar2cam
    input['cam2img'] = cam2img
    input['lidar2cam'] = lidar2cam
    input['lidar2img'] = lidar2img
    input['box_mode_3d'] = Box3DMode.LIDAR
    input['box_type_3d'] = LiDARInstance3DBoxes

# Print the output of this stage
if wanna_print_in_out: 
    print("\nAfter adding metainfos:")
    for element in inputs:
        print(element)






# TODO:
#   - Maybe add the gt_bboxes as well?

In [19]:
############################################### Pack3DDetInputs ###############################################

from mmdet3d.datasets.transforms.formating import Pack3DDetInputs

# Initialize the packer
packer = Pack3DDetInputs(keys=['points', 'img'])

# Print the input to this stage
if wanna_print_in_out: 
    print("\nPack3DDetInputs input:")
    for element in inputs:
        print(element)



# For cycle to also handle lists of inputs
for i in range(len(inputs)):
    
    # Actual modification of the dictionary
    inputs[i] = packer(inputs[i])



# Print the output of this stage
if wanna_print_in_out: 
    print("\nPack3DDetInputs output:")
    for element in inputs:
        print(element)

In [20]:
############################################### Det3DDataPreprocessor ###############################################

from mmdet3d.models.data_preprocessors.data_preprocessor import Det3DDataPreprocessor

# Initialize the preprocessor
preprocessor = Det3DDataPreprocessor(
    voxel=config_dictionary['Det3DDataPreprocessor']['voxel'],
    voxel_type=config_dictionary['Det3DDataPreprocessor']['voxel_type'],
    voxel_layer=config_dictionary['Det3DDataPreprocessor']['voxel_layer'],
    mean=config_dictionary['Det3DDataPreprocessor']['mean'],
    std=config_dictionary['Det3DDataPreprocessor']['std'],
    bgr_to_rgb=config_dictionary['Det3DDataPreprocessor']['bgr_to_rgb'],
    pad_size_divisor=config_dictionary['Det3DDataPreprocessor']['pad_size_divisor']
)

# Print the input to this stage
if wanna_print_in_out: 
    print("\nDet3DDataPreprocessor input:")
    for element in inputs:
        print(element)



# Create the list with the final inputs
final_inputs = []
for i in range(len(inputs)):    

    # Create a temporary dictionary to be passed to the preprocessor (in the right format)
    temp = {
        'data_samples': [inputs[i]['data_samples']],
        'inputs': inputs[i]['inputs']}
    temp['inputs']['points'] = [inputs[i]['inputs']['points']]
    temp['inputs']['img'] = [inputs[i]['inputs']['img']]

    # Take out the result of the Det3DDataPreprocessor, and also compute the time
    start_preprocessing = time.time()
    final_inputs.append(
        preprocessor(temp))
    torch.cuda.synchronize()
    end_preprocessing = time.time()
    delta_preprocessing.append(end_preprocessing-start_preprocessing)



# Move the tensors to the right device
for element in final_inputs:
    element['inputs']['points'][0] = element['inputs']['points'][0].to('cuda:0')
    element['inputs']['voxels']['voxels'] = element['inputs']['voxels']['voxels'].to('cuda:0')
    element['inputs']['voxels']['coors'] = element['inputs']['voxels']['coors'].to('cuda:0')
    element['inputs']['imgs'] = element['inputs']['imgs'].to('cuda:0')

# Print the output of this stage
if wanna_print_in_out: 
    print("\nDet3DDataPreprocessor output:")
    for element in final_inputs:
        print(element)

In [21]:
################################################# JUST Prediction #################################################

# predictions=[]
# for element in final_inputs:
#     # Take out the result of the inferencer
#     predictions.append(
#         model.predict(element['inputs'], element['data_samples'])
#     )



# # Print results of the predictions
# for i, element in enumerate(predictions):
#     print(f"\n\n\n--------------------------------\nPointcloud and Image {i}:")
#     whole_tensor = element[0].pred_instances_3d.bboxes_3d.tensor
#     for j in range(whole_tensor.shape[0]):
#         print(f"\n--------Instance {j+1}:\n{whole_tensor[j]}")

In [22]:
################################################# Model Step-by-Step #################################################

for i, element in enumerate(final_inputs):

    # Unpacking of the important elements
    batch_input_metas = [element['data_samples'][0].metainfo]
    voxel_dict = element['inputs'].get('voxels', None)
    imgs = element['inputs'].get('imgs', None)
    points = element['inputs'].get('points', None)

    # Image features extraction
    start_image_feats = time.time()
    img_feats = model.extract_img_feat(imgs, batch_input_metas)
    torch.cuda.synchronize()
    end_image_feats = time.time()
    delta_image_features.append(end_image_feats-start_image_feats)

    # Points --> DynamicVFE and PointFusion
    start_pointfusion = time.time()
    voxel_features, feature_coors = model.pts_voxel_encoder(voxel_dict['voxels'], voxel_dict['coors'], points, img_feats, batch_input_metas)
    torch.cuda.synchronize()
    end_pointfusion = time.time()
    delta_point_fusion.append(end_pointfusion-start_pointfusion)

    # Points --> SparseEncoder, SECOND, SECONDFPN
    start_pointencoder = time.time()
    batch_size = voxel_dict['coors'][-1, 0] + 1
    x = model.pts_middle_encoder(voxel_features, feature_coors, batch_size)
    x = model.pts_backbone(x)
    pts_feats = model.pts_neck(x)
    torch.cuda.synchronize()
    end_pointencoder = time.time()
    delta_point_encoder.append(end_pointencoder-start_pointencoder)

    # Postprocessing
    start_postprocessing = time.time()
    results_list_3d = model.pts_bbox_head.predict(pts_feats, element['data_samples'])
    torch.cuda.synchronize()
    end_postprocessing = time.time()
    delta_postprocessing.append(end_postprocessing-start_postprocessing)

    # Clean variables to avoid OutOfMemoryError
    img_feats = voxel_features = feature_coors = x = pts_feats = None

In [23]:
#################################################################################################################
#                                            PLOTTING OF FREQUENCY                                              #
#################################################################################################################

from demo.plotters import *

# Check that the vectors are all of the same dimension
if len(delta_preprocessing) != len(delta_image_features) or len(delta_preprocessing) != len(delta_point_fusion) or \
    len(delta_preprocessing) != len(delta_point_encoder) or len(delta_preprocessing) != len(delta_postprocessing):
    print("\nWrong dimensions for lists!!!\n")
    exit()



# Create the vector with the TOTAL delta_time
delta_total = []
for i in range(len(delta_preprocessing)):
    delta_total.append(delta_preprocessing[i]+delta_image_features[i]+delta_point_fusion[i]+delta_point_encoder[i]+delta_postprocessing[i])



# Create the vector with the INFERENCE delta_time
delta_inference = []
for i in range(len(delta_image_features)):
    delta_inference.append(delta_image_features[i]+delta_point_fusion[i]+delta_point_encoder[i])

In [ ]:
# Plot the total time

freq_plot_with_gaussian(delta_total, "Total time", "blue")

In [ ]:
# Plot the pre-processing time

freq_plot_with_gaussian(delta_preprocessing, "Pre-processing time", "green")

In [ ]:
#Plot the inference time

freq_plot_with_gaussian(delta_inference, "Network Inference time", "gold")

In [ ]:
# Plot the pie chart with the BIG PICTURE percentages

plot_pie_chart(delta_preprocessing, delta_inference, delta_postprocessing, "Pre-processing", "Inference", "Post-processing")

In [ ]:
# Plot the pie chart with the SMALL PICTURE percentages

plot_pie_chart(delta_image_features, delta_point_fusion, delta_point_encoder, "Image features", "Point Fusion", "Point Encoder")